In [ ]:
import logging
import time
from datetime import datetime, timezone

import pandas as pd

URL = "https://www.ianseo.net/TourData/2026/27297/IQAL.php"
OUTPUT_CSV = "brit_results.csv"
POLL_SECONDS = 120


logging.basicConfig(
	level=logging.INFO,
	format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)


def _normalize_name(name: str) -> str:
	"""Normalize column names so matching works across minor table variations."""
	return "".join(ch for ch in str(name).lower() if ch.isalnum())


def _find_column(columns: list[str], candidates: list[str]) -> str | None:
	"""Return the first table column that matches one of the candidate names."""
	normalized = {_normalize_name(col): col for col in columns}
	for candidate in candidates:
		hit = normalized.get(_normalize_name(candidate))
		if hit:
			return hit
	return None


def _to_numeric_score(series: pd.Series) -> pd.Series:
	"""Convert values to numeric, tolerating score strings like '560/600'."""
	extracted = series.astype(str).str.extract(r"^\s*(\d+)(?:\s*/.*)?$", expand=False)
	return pd.to_numeric(extracted, errors="coerce")

def _to_numeric(series: pd.Series) -> pd.Series:
	extracted = series.astype(str).str.extract(r"(\d+)", expand=False)
	return pd.to_numeric(extracted, errors="coerce")


def fetch_aggregates(url: str) -> dict:
	"""Fetch the IQAL results table and return summed Score, Hits, and Golds."""
	tables = pd.read_html(url, flavor="lxml")
	if not tables:
		raise ValueError(f"No tables found at {url}")

	df = tables[0].dropna(how="all").copy()
	if isinstance(df.columns, pd.MultiIndex):
		df.columns = [str(c).strip() for c in df.columns.get_level_values(-1)]
	else:
		df.columns = [str(c).strip() for c in df.columns]

	score_col = _find_column(df.columns.tolist(), ["Tot.", "Tot", "Score", "Total"])
	hits_col = _find_column(df.columns.tolist(), ["Hits", "Hit"])
	golds_col = _find_column(df.columns.tolist(), ["Golds", "Gold"])

	if not score_col:
		raise ValueError("Could not find a score column in the scraped table.")

	score_total = float(_to_numeric_score(df[score_col]).sum(skipna=True))
	hits_total = float(_to_numeric(df[hits_col]).sum(skipna=True)) if hits_col else 0.0
	golds_total = float(_to_numeric(df[golds_col]).sum(skipna=True)) if golds_col else 0.0

	return {
		"timestamp_utc": datetime.now(timezone.utc).isoformat(),
		"score_total": score_total,
		"hits_total": hits_total,
		"golds_total": golds_total,
		"rows_scraped": int(len(df)),
	}


def load_history(path: str) -> pd.DataFrame:
	"""Load previous poll results if they exist, otherwise start empty."""
	try:
		return pd.read_csv(path)
	except FileNotFoundError:
		return pd.DataFrame(
			columns=[
				"timestamp_utc",
				"score_total",
				"hits_total",
				"golds_total",
				"rows_scraped",
			]
		)


def run_polling_loop() -> None:
	"""Poll every 10 minutes, append to DataFrame, and persist as CSV."""
	history = load_history(OUTPUT_CSV)
	logger.info("Starting polling loop. Writing results to %s", OUTPUT_CSV)

	while True:
		try:
			aggregates = fetch_aggregates(URL)
			history = pd.concat([history, pd.DataFrame([aggregates])], ignore_index=True)
			history.to_csv(OUTPUT_CSV, index=False)
			logger.info(
				"Saved row #%d | score=%s hits=%s golds=%s",
				len(history),
				aggregates["score_total"],
				aggregates["hits_total"],
				aggregates["golds_total"],
			)
		except Exception:
			logger.exception("Polling iteration failed")

		time.sleep(POLL_SECONDS)


if __name__ == "__main__":
	run_polling_loop()


2026-03-22 11:25:51,715 - INFO - Starting polling loop. Writing results to brit_results.csv
2026-03-22 11:25:52,353 - INFO - Saved row #120 | score=92699.0 hits=11523.0 golds=2506.0
2026-03-22 11:27:52,817 - INFO - Saved row #121 | score=92699.0 hits=11523.0 golds=2506.0
2026-03-22 11:29:53,183 - INFO - Saved row #122 | score=92747.0 hits=11529.0 golds=2506.0
2026-03-22 11:31:53,522 - INFO - Saved row #123 | score=93830.0 hits=11649.0 golds=2560.0
2026-03-22 11:33:54,029 - INFO - Saved row #124 | score=93830.0 hits=11649.0 golds=2560.0
2026-03-22 11:35:54,416 - INFO - Saved row #125 | score=94570.0 hits=11739.0 golds=2584.0
2026-03-22 11:37:54,760 - INFO - Saved row #126 | score=94570.0 hits=11739.0 golds=2584.0
2026-03-22 11:39:56,252 - INFO - Saved row #127 | score=95107.0 hits=11808.0 golds=2602.0
2026-03-22 11:41:56,604 - INFO - Saved row #128 | score=95107.0 hits=11808.0 golds=2602.0
2026-03-22 11:43:57,119 - INFO - Saved row #129 | score=95409.0 hits=11847.0 golds=2608.0
2026-03-